# 🔥 Proposed Model 2 — TherA + SAM2 Thermal Target Locker\n\n**Pipeline:** RGB video → TherA converts all frames → `thermal.mp4` → display thermal frame 1 → mouse click target → SAM2 mask → track all thermal frames → `output_thermal_tracking.mp4`\n\nUse a **GPU runtime** in Google Colab. TherA produces synthetic TIR imagery, not measured temperature data.\n

In [ ]:
import os, sys, subprocess, urllib.request, shutil
from pathlib import Path
import torch

if not torch.cuda.is_available():
    raise RuntimeError("GPU not detected. Colab: Runtime → Change runtime type → T4 GPU.")

REPO = Path("/content/Target_Locker")

# Always use the merged main branch.
if not REPO.exists():
    subprocess.run([
        "git","clone","--depth","1","--branch","main",
        "https://github.com/CptImtiaz/Target_Locker.git", str(REPO)
    ], check=True)
else:
    subprocess.run(["git","-C",str(REPO),"fetch","origin","main"], check=True)
    subprocess.run(["git","-C",str(REPO),"checkout","main"], check=True)
    subprocess.run(["git","-C",str(REPO),"pull","--ff-only","origin","main"], check=True)

subprocess.run([
    sys.executable,"-m","pip","install","-q","-U","huggingface_hub"
], check=True)

subprocess.run([
    sys.executable,"-m","pip","install","-q",
    "-r",str(REPO/"proposed_model_2"/"requirements.txt")
], check=True)

THERA = Path("/content/TherA")
if not THERA.exists():
    subprocess.run([
        "git","clone","--depth","1",
        "https://github.com/donkeymouse/TherA.git", str(THERA)
    ], check=True)
else:
    subprocess.run(["git","-C",str(THERA),"pull","--ff-only"], check=True)

subprocess.run([
    sys.executable,"-m","pip","install","-q",
    "-r",str(THERA/"requirements.txt")
], check=True)

# ------------------------------------------------------------------
# TherA weights: use the Hugging Face Python API instead of the legacy
# huggingface-cli command. Only download files needed by reference-cache
# inference (no full LLaVA runtime model).
# ------------------------------------------------------------------
from huggingface_hub import snapshot_download

weights = THERA / "weights"
weights.mkdir(parents=True, exist_ok=True)

required = [
    weights / "checkpoint" / "model.pt",
    weights / "merged_models",
    weights / "stable-diffusion",
]

if not all(p.exists() for p in required):
    print("⬇️ Downloading TherA inference weights from Hugging Face...")
    snapshot_download(
        repo_id="donkeymouse/TherA",
        local_dir=str(weights),
        allow_patterns=[
            "model.pt",
            "checkpoint/*",
            "checkpoint/**",
            "merged_models/*",
            "merged_models/**",
            "stable-diffusion/*",
            "stable-diffusion/**",
            "reference_caches/*",
            "reference_caches/**",
            "*SUNNY.pt",
            "**/SUNNY.pt",
        ],
    )

# Current TherA code expects weights/checkpoint/model.pt.
# Some published weight layouts put model.pt directly under weights/.
checkpoint_dir = weights / "checkpoint"
checkpoint_dir.mkdir(parents=True, exist_ok=True)
checkpoint_model = checkpoint_dir / "model.pt"
root_model = weights / "model.pt"

if not checkpoint_model.exists() and root_model.exists():
    shutil.copy2(root_model, checkpoint_model)

if not checkpoint_model.exists():
    # Last-resort search in case Hugging Face changes the folder layout.
    candidates = [p for p in weights.rglob("model.pt") if p.is_file()]
    if candidates:
        shutil.copy2(candidates[0], checkpoint_model)

# Locate SUNNY cache and normalize its location.
cache_dir = weights / "reference_caches"
cache_dir.mkdir(parents=True, exist_ok=True)
sunny_cache = cache_dir / "SUNNY.pt"

if not sunny_cache.exists():
    candidates = [p for p in weights.rglob("SUNNY.pt") if p.is_file()]
    if candidates:
        shutil.copy2(candidates[0], sunny_cache)

missing = []
if not checkpoint_model.exists():
    missing.append(str(checkpoint_model))
if not (weights / "merged_models").exists():
    missing.append(str(weights / "merged_models"))
if not (weights / "stable-diffusion" / "vae").exists():
    missing.append(str(weights / "stable-diffusion" / "vae"))
if not (weights / "stable-diffusion" / "scheduler").exists():
    missing.append(str(weights / "stable-diffusion" / "scheduler"))
if not sunny_cache.exists():
    missing.append(str(sunny_cache))

if missing:
    print("\nTherA files found:")
    for p in sorted(weights.rglob("*")):
        if p.is_file():
            print(" -", p.relative_to(weights))
    raise RuntimeError(
        "TherA download completed but required inference files are missing:\n"
        + "\n".join(f" - {m}" for m in missing)
        + "\n\nThe upstream TherA weight repository may not currently include the "
          "reference cache files. The rest of Proposed Model 2 is installed correctly."
    )

SAM_CKPT = Path("/content/sam2.1_hiera_tiny.pt")
if not SAM_CKPT.exists():
    urllib.request.urlretrieve(
        "https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_tiny.pt",
        SAM_CKPT
    )

sys.path.insert(0, str(REPO/"proposed_model_2"))

print("✅ Proposed Model 2 ready")
print("TherA checkpoint:", checkpoint_model)
print("TherA SUNNY cache:", sunny_cache)
print("SAM2 checkpoint:", SAM_CKPT)



In [ ]:
import os, sys
from pathlib import Path

REPO = Path("/content/Target_Locker")
sys.path.insert(0, str(REPO/"proposed_model_2"))
os.chdir(REPO/"proposed_model_2")

exec((REPO/"proposed_model_2"/"app_interface.py").read_text(), globals())
\n